In [0]:
from pyspark.sql.functions import col

quality_df = spark.table("workspace.default.gold_quality_by_table")
rule_df = spark.table("workspace.default.gold_quality_by_rule")

def get_worst_quality_table():
    row = (
        quality_df
        .orderBy(col("quality_score").asc())
        .limit(1)
        .collect()[0]
    )
    
    return {
        "table_name": row["table_name"],
        "quality_score": row["quality_score"],
        "failed_rules": row["failed_rules"],
        "total_rules": row["total_rules"]
    }

def get_failed_rules(table_name):
    rows = (
        rule_df
        .filter(col("table_name") == table_name)
        .orderBy(col("total_failure_percentage").desc())
        .collect()
    )
    
    return [
        {
            "rule": r["rule"],
            "rule_count": r["rule_count"],
            "total_failure_percentage": r["total_failure_percentage"]
        }
        for r in rows
    ]

def generate_remediation_plan(table_name):
    failed_rules = get_failed_rules(table_name)
    
    recommendations = []
    
    for item in failed_rules:
        rule = item["rule"]
        
        if rule == "not_null":
            recommendations.append("Add mandatory field checks at ingestion and reject records with missing critical values.")
        elif rule == "unique":
            recommendations.append("Add deduplication logic using business keys before writing to Silver/Gold tables.")
        elif rule == "regex":
            recommendations.append("Validate format using regex rules before loading into curated tables.")
        elif rule == "accepted_values":
            recommendations.append("Add domain validation and maintain a reference table for allowed values.")
        else:
            recommendations.append(f"Review rule {rule} and define a business-specific remediation.")
    
    return recommendations

worst_table = get_worst_quality_table()
print("Worst quality table:")
print(worst_table)

print("\nFailed rules:")
print(get_failed_rules(worst_table["table_name"]))

print("\nRemediation plan:")
print(generate_remediation_plan(worst_table["table_name"]))